# fz-Telemac Plugin Example Usage

This notebook demonstrates how to use the fz-Telemac plugin.

The examples cover:
1. Installing dependencies (fz framework and fz-Telemac plugin)
2. Downloading example input files
3. Parsing input files to identify variables
4. Compiling input files with specific parameter values
5. Running calculations with Telemac
6. Performing parametric studies

**Note:** This plugin requires a Telemac installation (local or via Docker).

## Setup: Install Dependencies

First, install the Funz framework and the fz-Telemac plugin.

In [ ]:
# Install fz framework from GitHub
%pip install -q git+https://github.com/Funz/fz.git

# Install fz-Telemac plugin
import fz
fz.install("Telemac")

print("✓ Dependencies installed successfully!")

## Setup: Download Example Files

Download the example input files from the GitHub repository.

In [ ]:
import os
import urllib.request

# Create directory for examples
os.makedirs("Telemac/t2d_breach.cas", exist_ok=True)

# Base URL for raw GitHub content
base_url = "https://raw.githubusercontent.com/Funz/fz-Telemac/main/examples/Telemac/t2d_breach.cas"

# Download example files
files_to_download = [
    ("Telemac/t2d_breach.cas/t2d_breach.cas", f"{base_url}/t2d_breach.cas"),
    ("Telemac/t2d_breach.cas/breach.txt", f"{base_url}/breach.txt"),
    ("Telemac/t2d_breach.cas/t2d_breach.poi", f"{base_url}/t2d_breach.poi"),
    ("Telemac/t2d_breach.cas/t2d_breach.liq", f"{base_url}/t2d_breach.liq"),
    ("Telemac/t2d_breach.cas/geo_breach.cli", f"{base_url}/geo_breach.cli"),
]

print("Downloading example files...")
for local_path, url in files_to_download:
    try:
        urllib.request.urlretrieve(url, local_path)
        print(f"  ✓ Downloaded: {local_path}")
    except Exception as e:
        print(f"  ✗ Failed to download {local_path}: {e}")

print("\n✓ Example files ready!")

## Import Required Libraries

In [ ]:
import fz
import os
import tempfile

print(f"fz version: {fz.__version__ if hasattr(fz, '__version__') else 'unknown'}")

## Example 1: Parse Input File for Variables

Use `fz.fzi()` to identify variables in an input file.

In [ ]:
# Parse the input file to find variables
variables = fz.fzi(
    "Telemac/t2d_breach.cas/",
    "Telemac"
)

print(f"Variables found in t2d_breach.cas: {list(variables.keys())}")

## Example 2: Compile Input Files

Use `fz.fzc()` to compile input files with specific parameter values. This substitutes variables with actual values.

In [ ]:
# Create temporary directory for compiled files
with tempfile.TemporaryDirectory() as tmpdir:
    # Compile input with specific values (empty dict if no variables)
    fz.fzc(
        "Telemac/t2d_breach.cas/",
        {},  # Add your variable values here, e.g., {"breach_width": 50.0}
        "Telemac",
        output_dir=tmpdir
    )
    
    print("Compiled files created in:", tmpdir)
    print("✓ Compilation successful")

## Example 3: Run a Basic Calculation

Run a simple Telemac calculation.

**Note:** This requires Telemac to be installed (via Docker or locally). Make sure the calculator script is properly configured.

In [ ]:
# Define parameter values (empty if base case)
input_variables = {}

try:
    # Run calculation
    results = fz.fzr(
        input_path="Telemac/t2d_breach.cas/",
        input_variables=input_variables,
        model="Telemac",
        calculators="localhost_Telemac",
        results_dir="results/basic_study"
    )
    
    # Display results
    print("\nResults:")
    print("=" * 70)
    print(results)
    
except Exception as e:
    print(f"\nError running calculation: {e}")
    print("Note: Make sure Telemac is properly installed and configured.")

## Example 4: Multi-Parameter Study

Demonstrate a parametric study with multiple variables.

First, you need to add variables to your input files. For example, edit the breach.txt file:
```bash
sed -i 's/50.0/$(breach_width)/' Telemac/t2d_breach.cas/breach.txt
sed -i 's/300.0/$(breach_delay)/' Telemac/t2d_breach.cas/breach.txt
```

In [ ]:
# Define multiple parameters
input_variables = {
    "breach_width": [40, 50, 60],
    "breach_delay": [200, 300]
}

print(f"Running parametric study...")
print(f"  breach_width values: {input_variables['breach_width']}")
print(f"  breach_delay values: {input_variables['breach_delay']}")

try:
    # Run parametric calculation
    results = fz.fzr(
        input_path="Telemac/t2d_breach.cas/",
        input_variables=input_variables,
        model="Telemac",
        calculators="localhost_Telemac",
        results_dir="results/multi_param_study"
    )
    
    # Display results
    print("\nResults:")
    print("=" * 70)
    print(results)
    
except Exception as e:
    print(f"\nError running parametric study: {e}")
    print("Note: Make sure the input files contain the variable placeholders.")

## Example 5: Visualization

If you have results, you can visualize them using matplotlib.

In [ ]:
import matplotlib.pyplot as plt

# Example visualization of water level over time
try:
    # Use actual results if available
    if 'results' in dir() and results is not None and 'S' in results:
        plt.figure(figsize=(12, 6))
        
        # Plot results for each parameter combination
        for i in range(len(results['S'])):
            if 't2d_breach' in results['S'][i]:
                data = results['S'][i]['t2d_breach']
                if 'time' in data and 'xylowercenter' in data:
                    time = list(data['time'].values())
                    water_level = list(data['xylowercenter'].values())
                    label = f"Run {i+1}"
                    if 'breach_width' in results:
                        label = f"Width: {results['breach_width'][i]}, Delay: {results['breach_delay'][i]}"
                    plt.plot(time, water_level, '-o', label=label, alpha=0.7)
        
        plt.xlabel('Time (s)', fontsize=12)
        plt.ylabel('Water level (m)', fontsize=12)
        plt.title('Water Level Over Time at Lower Center Point', fontsize=14, fontweight='bold')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
    else:
        print("No results available for visualization.")
        print("Run a calculation first to generate results.")
    
except Exception as e:
    print(f"Visualization error: {e}")